## 1.0 Introduction

This notebook is designed to provide an accesible platform for analysing sequencing data. This notebook allows you to perfom analysis without the need to dowload or install any additional software on your computer. You only need is a Google account.

Before you can start with this workshop, you need to connect this notebook document with our Google Drive. This step is crucial because it allows you to import the example data to Google drive, and later on, save output files directly to Google Drive. Without this step, any files created during this Colab session will be lost once the Colab environment is closed (or connection to it is lost).

The text below is code in text block that is called a code block or code chunk. It is advised not to change text in the code blocks. The code can be executed by clicking the 'play' button in the top left corner. In this workshop, each code block will be introduced with a brief statement describing what analysis step will be performed by the code.

For example, by the code block below you will install the rpy2 library, which is a software package that allows you to run R scripts directly within the Python environment of this notebook. This enables an integration of R’s statistical and microbiome analysis functions required for diving into the water microbiome.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install rpy2
%load_ext rpy2.ipython
!git clone https://github.com/stijnteunissen/Workshop_H2Omics_test.git

# 2.0 Analysis Preparation

### 2.1 Creating the Output Folder

This code block creates a directory in Google Drive within the home folder (`"drive/MyDrive/"`). The new directory will be created after choosing a self-defined project name, and then uses that to name for the directory. This directory will be used to store all output files generated during the workshop, ensuring they are organized and saved in your Google Drive. After running the code, write a project name (use underscores "_" instead of spaces), and press confirm for the project name.

In [ ]:
import sys
sys.path.append("/content/Workshop_H2Omics_test/H2Omics_workshop")

from starting_project import starting_project

starting_project()

### 2.2 Loading Input Files

In this code block, you will be asked to upload the required microbiome data files from your local computer. An extended notebook version complements this concise workshop version. The extended notebook enables you to independently execute any preceding QIIME2 code blocks, in order to produce the four required data files. For this workshop, we will import example files produced by using QIIME2.

The four files include:

- **Feature Table:** This data matrix contains the sequence read counts of all microbial features (e.g., OTUs or ASVs) for all the samples in sequencing project.
- **Taxonomy Table:** This table contains the hierarchical taxonomic information (Kingdom, Phylum, Class, Order, Family, Genus, Species) that each microbial feature is classified to. In this workshop, the classification is based on SILVA reference database version 138.1.
- **Phylogenetic Tree:** The phylogeny represents the evolutionary relationships among the microbial features based on their DNA sequence similarity.
- **Sample Metadata:** This table includes information describing the project's samples in detail, including sampling date, experimental factors and additional measurements (e.g., pH, DNA concentration, dilution factors, chemical data, etc.).

Uploading these files allows the notebook to integrate all the required data for a comprehensive analysis of your sequencing dataset.
The code block below also allows your to add additional data files for the same samples. While such data could already have been included in the Sample Metadata, the code block below allows adding extra data that will be combined with the Sample Metadata. In the context of microbiome analysis, a specific example is biomass data that can be used to integrate information on 'microbial load'. Two explicit options typically applied in the water context are made possible in the workshop:
- **Flow cytometry** based cell concentration data
- **qPCR** based 16S rRNA gene copy concentration data

NOTE in additional data files, Sample IDs should match identically with Sample Metadata.

All required data for this workshop are readily available and no additional files are required to upload.

Below, choose which type of biomass data to apply for normalisation of microbiome sequence data. Choose fcm (flow cytometry) or qpcr, and confirm the choice.

In [ ]:
import sys
sys.path.append("/content/Workshop_H2Omics_test/H2Omics_workshop")

from norm_method import select_norm_method

select_norm_method()

In [ ]:
import sys
sys.path.append("/content/Workshop_H2Omics_test/H2Omics_workshop")
from import_files import import_files

import_files()

### 2.3 Installing and Loading R Packages

In this section, you install and load all the required R packages required or useful for performing microbiome sequencing data analysis.

In [ ]:
import sys
sys.path.append("/content/Workshop_H2Omics_test/H2Omics_workshop")

from install_conda import install_conda

install_conda()

In [ ]:
%%R
source("/content/Workshop_H2Omics_test/H2Omics_workshop/install_packages.R")

install_packages()

# 3.0 Starting the analysis

### 3.1 copy number prediction
The analysis starts specifically with using the DNA sequence representative for each of the amplicon sequence variants (ASVs), to predict the number of ribosomal RNA gene copies present in the genome of the bacterium the ASV belongs to. The output is a table in which for each ASV (column 'OTU') the predicted gene copy number is included (as well as a confidence probability).

In [ ]:
%%R
source("/content/Workshop_H2Omics_test/H2Omics_workshop/copy_number_prediction.R")

copy_number_prediction()

### 3.2 Creating a Logging System

To improve the reproducibility of your microbiome data analysis, this code block creates a directory "messages" in which a log file is automatically updated (logging_output.txt), from here onwards, tracking every step in the analysis with time stamps.

In [ ]:
%%R
dir.create(paste0(base_path, projects, "/messages"), recursive = TRUE)
log_file = paste0(base_path, glue("{projects}/messages/logging_output.txt"))

log_message = function(message, log_file) {
  write(paste(format(Sys.time(), "%Y-%m-%d %H:%M:%S"), "-", message), log_file, append = TRUE)
}

### 3.3 Setting up the project structure

The code block below creates a directory structure for each project, ensuring that all necessary directories exist and that specific files required for downstream analysis are (with the extended notebook, copied from the `qiime2_output` folder) into the `input_data` folder.

### 3.4 Merging Metadata

This code block also merges and reformats metadata, combining
QIIME metadata with potentially additionally uploaded sample metadata, such as qPCR data or FCM data, creating a single unified metadata file for downstream analyses.

In [ ]:
%%R
# create folders and copy files
H2Omics::create_folders(projects)

log_message(paste("Step 1: Merge metadata: adding fcm or qpcr data to the original metadata.", paste(projects, collapse = ", ")), log_file)

# combine qiime2 metadata with experimental sampled metadata
unified_metadata = H2Omics::unify_metadata(projects)

log_message("Metadata successfully merged.", log_file)

### 3.5 Combining all data into a phyloseq object

A Phyloseq object is an R data structure that combines the four essential data files, the feature table, the taxonomy table, phylogenetic tree and unified sample metadata, into a single coherent dataset for streamlined analysis. The code block below creates a `phyloseq` object from the data you loaded into this notebook.


In [ ]:
%%R
log_message(paste("Step 2: Creating phyloseq object (physeq): is created using the table, rooted tree, classifier and metadata.", paste(projects, collapse = ", ")), log_file)

# create a physeq object
physeq = H2Omics::creating_physeq_object(projects)

log_message("Phyloseq successfully created.", log_file)

### 3.6 Optimizing the taxonomic information
This function cleans and filters the taxonomy table within a `phyloseq` object. It removes unclassified or ambiguous names, and replaces these and missing taxon names at genus level with placeholders derived from higher
taxonomic ranks. For example, if for a given Enterobacteriaceae ASV the genus name could not be classified with a certain minimum confidence, the empty cell is filled with "Genus of Enterobacteriaceae" instead of the universal 'unclassified'.

The `tax_filter` option of the function `tax_clean()` (TRUE or FALSE) can be specified to remove specific taxa, e.g., ASVs classified to the kingdom Eukaryota, the class of chloroplasts, the family of mitochondria, as well as ASVs unclassified at the Kingdom and Phylum levels.

The cleaned taxonomy is then ready to summarize the microbial abundances at higher taxonomic ranks, through the procedure of taxonomic agglomeration with the function (`tax_glom`) to, for example, the genus level. The updated taxonomic information now prevents merging 'unclassified' ASVs from diverse phylogenetic ancestry into a single artifical genus called 'unclassified' or 'ambiguous taxon'.

The code block below first shows the first ten rows of the taxonomic table of the uncleaned phyloseq object, and then shows the taxonomic table after cleaning it up.

In [ ]:
%%R
log_message(paste("Step 3: Tax clean: phyloseq taxa are cleaned.", paste(projects, collapse = ", ")), log_file)

# tax clean
cleaned_physeq = H2Omics::tax_clean(physeq = physeq, tax_filter = TRUE)

log_message("Successfully Tax cleaned.", log_file)

### 3.7 Resolving the phylogenetic tree

This code block performs a similar cleaning step to your `phyloseq` object, but this time to the phylogenetic tree by resolving all nodes into bifurcations. The original tree is then replaced by the updated tree in the `phyloseq` object.

### 3.8 Removing contaminant ASVs
Also in the code block, an elaborate step is made to remove contaminating ASVs from the phyloseq object. This approach uses both the information gained from sequencing blanks (empty DNA samples that thus contain contaminants from the sample processing in the lab), and/or, uses a statistical model to predict ASVs to be contaminants by relating their abundance in samples with the DNA concentration of samples (which is recorded in the sample metadata). In short, contaminant ASVs are more prevalent and likely more abundant in samples with low DNA content. With high DNA conctrations (high biomass samples) contaminants will be outcompeted by true sample-derived DNA.

This decontamination process also generates figures that illustrate the read counts and prevalence of contaminants across the samples in the Project.

### 3.9 Removing Mock community ASVs
A self-created mock microbiota sample with known bacteria is sequenced along with the samples as a positive control. Any mock bacteria that cross-contaminated into the samples are to be removed. The last function in the code block removes mock ASVs (and the mock sample) from your phyloseq object.

In [ ]:
%%R
log_message(paste("Step 4: Resolving tree.", paste(projects, collapse = ", ")), log_file)

# Resolve polytomous branching of the QIIME2 Fasttree2 phylogeny into a fully bifurcated tree for phylogenetic analyses
resolved_tree_physeq = H2Omics::resolve_tree(physeq = cleaned_physeq)

log_message("Tree successfully resolved", log_file)

log_message(paste("Step 5: Decontam: Removing contamination.", paste(projects, collapse = ", ")), log_file)

# decontam (decon_method = frequency, prevalence or both)
decontam_physeq = H2Omics::decontam(physeq = resolved_tree_physeq, decon_method = both, blank = TRUE)

log_message("Decontam successfully executed.", log_file)

log_message(paste("Step 6: Removing mock: Mock samples en mock OTUs are removed fropm phyloseq object.", paste(projects, collapse = ", ")), log_file)

# remove mock and mock features
without_mock_physeq = H2Omics::remove_mock(physeq = decontam_physeq, mock_genera = mock_genera, mock = TRUE)

log_message("Mock successfully removed.", log_file)

### 3.10 Ribosomal gene copy number correction

Microbiome analysis is typically performed by sequencing 16S ribosomal RNA fragments, amplified first by PCR. However, the number of copies of 16S rRNA gene copies varies among bacterial species, ranging 1-15 copies per genome.
After your phyloseq object has been cleaned from potential contamination, three additional steps can be taken to enhance meaningful interpretation of your microbiome data.

In the code block below, first, a copy number correction is applied to your `phyloseq` object. The sequence count of every ASV in every sample is divided by the predicted number of ribosomal gene copies for that ASV, which you have predicted with the `copy_number_prediction()` function at the start of the analysis. This copy number correction reduces an overestimation of the abundance of bacteria that have more rRNA gene copies than average, and the opposite is true for bacteria with fewer than average ribosomal RNA gene copies.

Going forward, the resulting unit of analysis will change from sequence count to approximated cell count (per unit sample, such as per liter or per sample), which could be more meaningful in a public health context. The choice is yours to apply this correction. Choose TRUE or FALSE, and then confirm your choice.


In [ ]:
import sys
sys.path.append("/content/Workshop_H2Omics_test/H2Omics_workshop")

from copy_correction import select_copy_correction

select_copy_correction()

### 3.11 Biomass normalisation of microbiome

The second step before you will have a proper look at your microbiome data, you will combine the microbiome data with paired measurement of biomass of the sequenced samples. This normalization step is able to account for the biomass of each sample before visualization or statistical hypothesis testing. This biomass normalization helps obtaining more meaningful interpretation of microbiome data. For example, comparing a sample with $10^{3}$ cells·mL$^{-1}$ with a sample with $10^{6}$ cells·mL$^{-1}$ with identical relative contributions of all bacteria, will be different after the factor 1000 difference is integrated by the normalization step. The normalized compositions will more accurately reflect the true microbial composition of the sampled ecosystem.
In the data import stage earlier on, you have selected which type of biomass data you will use, either flow cytometry (FCM) or qPCR data.

In [ ]:
%%R
log_message(paste("Step 7: copy number correction for relative data and biomass normalisation for absolute data.", paste(projects, collapse = ", ")), log_file)

# copy number correction and biomasss normalisation
normalised_asv_physeq = H2Omics::normalise_data(physeq = without_mock_physeq, norm_method = norm_method, copy_correction = copy_correction)

log_message("Anna16 correction and fcm or qpcr normalisation successfully modified.", log_file)

### 3.12 Rarefaction

Lastly, rarefaction is the process random subsampling an equal number of units from every sample. Comparing microbiomes after rarefaction makes sure to interpret microbiome differences in an unbiased way.  Rarefaction is applied on the normalized `phyloseq` object created in the previous step. How much to subsample during rarefaction is determined by the biomass of each sample and the number of sequences obtained from each sample. These are combined to define the minimum sampling depth across the dataset. In this way, you ensure that the results are unbiased to sampling depth (i.e. how much of the ecosystem have I sampled) and for sequencing depth (i.e. more sequences per sample = more depth).

### 3.13 Grouping data by taxonomy level

The code below also provides the option to aggregate your `phyloseq` object to a higher taxonomic level (i.e.,
Phylum, Class, Order, Family, and Genus) by summing counts using the `tax_glom()` function. Depending on the specified normalization method, the function
processes and saves both copy number corrected data and normalized data (using
flow cytometry or qPCR).

In [ ]:
%%R
log_message(paste("Step 8: rarefied data", paste(projects, collapse = ", ")), log_file)

# rarefied data
rarefied_asv_physeq = H2Omics::rarefying(physeq = normalised_asv_physeq, norm_method = norm_method, iteration = 10)

log_message("Data is successfully rarefied.", log_file)

log_message(paste("Step 9: Tax glom: OTUs are merged at different taxonomic levels.", paste(projects, collapse = ", ")), log_file)

rarefied_tax_physeq = H2Omics::group_tax(physeq = rarefied_asv_physeq, norm_method = norm_method)

log_message("Successfully Tax glom.", log_file)

log_message(paste("Step 10: Convert phyloseq to tibble.", paste(projects, collapse = ", ")), log_file)

# converting phyloseq object to a tibble
rarefied_tax_psmelt = H2Omics::psdata_to_tibble(physeq = rarefied_tax_physeq, norm_method = norm_method)

log_message("Successfully converted", log_file)

### 3.14 Creating a Barplot

The code block below facilitates generating barplots of microbiome data at the genus level (limited to the genus level, for simplicity). It supports visualizing relative abudances (ranging 0-100%) and normalized cell concentrations. You can now organize your barplots using experimental factors as supplied in the sample metadata. The resulting plots can be saved as PDF files, and the underlying data can be exported as MS Excel readable .CSV and R readable .RDS files.

In [ ]:
import sys
sys.path.append("/content/Workshop_H2Omics_test/H2Omics_workshop")

from factor_selection import factor_selection

factor_selection()

In [ ]:
%%R
log_message(paste("Step 11: Creating barplots.", paste(projects, collapse = ", ")), log_file)

# Relative Barplot
created_barplot = H2Omics::barplot(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method, sample_matrix = "liquid")

In [ ]:
%%R
# Absolute Barplot
created_barplot = H2Omics::barplot2(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method, sample_matrix = "liquid")

log_message("Barplot successfully plotted.", log_file)

In [ ]:
%%R
# pathogens genus
H2Omics::barplot_extra(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method, sample_matrix = "liquid")

In [ ]:
%%R
# pathogens genus
H2Omics::barplot_extra2(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method, sample_matrix = "liquid")

### 3.15 Creating a Heatmap

The barplots help assess differences among samples, but comparing bacteria is less intuitive. In the code block below, a heatmap is created visualizing the relative abundance data at the genus level. Similar to the barplots, bacteria that are below a defined abundance threshold are grouped into a category "other".


In [ ]:
%%R
log_message(paste("Step 12: making heatmap.", paste(projects, collapse = ", ")), log_file)

# heatmap
heatmap_plot = H2Omics::heatmap(physeq = rarefied_tax_psmelt, ntaxa = 23, norm_method = norm_method)

log_message("Heatmap successfully plotted.", log_file)

### 3.16 Alpha diversity: the bacterial richness and diversity within samples

The diversity of bacteria detected in multiple samples can be compared at the ASV level or at different taxonomic levels. Alpha diversity metrics can provide distinct aspects of this diversity, such as the observed number of taxa, the estimated total number of taxa (Chao1), or taking into account the degree of dominance of taxa (Shannon and Simpson indices). The function `estimate_richness()` calculates alpha diversity in the code block below. Depending on your previous choices, the function will use relative abundances or normalized count data.

In [ ]:
import sys
sys.path.append("/content/Workshop_H2Omics_test/H2Omics_workshop")

from taxrank_alpha_div import select_taxrank_alpha_div

select_taxrank_alpha_div()

In [ ]:
%%R
log_message(paste("Step 13: Making alpha diversity.", paste(projects, collapse = ", ")), log_file)

# alpha diversity
alpha_div_plots = H2Omics::alpha_diversity(physeq = rarefied_asv_physeq, taxrank = taxrank_alpha_div, norm_method = norm_method)

log_message("Alpha diversity successfully plotted.", log_file)

### 3.17 Beta diversity: pairwise comparing microbiome samples

Beta diversity describes the degree of clustering of samples and thus differentiation among groups of samples. Beta diversity can be calculated by pairwise comparison of microbiome compositions. An array of pairwise differences can be calculated. These options depend on taking as input only presence/absence (0/1) of bacteria (Jaccard index), the counts or relative abundances of detected taxa (Bray-Curtis index), or either of these options but also taking the relatedness among the taxa into account (UniFrac metric). Visualization of beta diversity is typically a principal coordinates analysis (PCoA) plot where x and y axes depict most of the sample clustering. Beta diversity can be calculated at all taxonomic levels, and using relative or normalised counts data. For the latter, another distance metric is suitable (Manhattan).


In [ ]:
import sys
sys.path.append("/content/Workshop_H2Omics_test/H2Omics_workshop")

from beta_diversity_options import select_beta_diversity_options

select_beta_diversity_options()

In [ ]:
%%R
log_message(paste("Step 14: Making beta diversity.", paste(projects, collapse = ", ")), log_file)

# beta diversity
beta_div_plots = H2Omics::beta_diversity(physeq = rarefied_asv_physeq, taxrank = taxrank_beta_div, norm_method = norm_method,
                                         ordination_method = "PCoA", color_factor = color_factor, color_continuous = FALSE,
                                         shape_factor = shape_factor, size_factor = NULL, alpha_factor = NULL)

log_message("Beta diversity successfully plotted.", log_file)

### 3.18 Exporting results and figures

The final code block of this workshop helps you to export results, output data and figures by creating a dedicated export folder within the project directory you created at the start of the workshop. It organises the export folder into subdirectories for figures, CSV files, and RDS files, and creates copies of the most relevant files from their original locations on your Google Drive.

In [ ]:
%%R
log_message(paste("Step 15: export figures and rds files.", paste(projects, collapse = ", ")), log_file)

# export data
H2Omics::export_data()

log_message("export completed.", log_file)